In this file, the embedding from Merfish data will be generated. Different from DLPFC, SOMDE will not be used here as there are less than 200 genes for Merfish.

In [1]:
import os
import torch
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse
import anndata
import random
from raftup import _gene_cost_somde


# =========================================================
# Setup
# =========================================================
def setup_environment(mode="reproducible", seed=0):
    """
    Configure random seeds and device.

    mode = "reproducible"
        - force CPU
        - enable deterministic algorithms
        - optionally limit thread-level nondeterminism

    mode = "fast"
        - prefer CUDA, then MPS, then CPU
        - allow nondeterministic behavior for speed
    """
    if mode not in {"reproducible", "fast"}:
        raise ValueError("mode must be 'reproducible' or 'fast'")

    # ----- reproducible mode -----
    if mode == "reproducible":
        # Optional: reduce CPU nondeterminism from parallel reductions
        os.environ["OMP_NUM_THREADS"] = "1"
        os.environ["MKL_NUM_THREADS"] = "1"

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        torch.set_num_threads(1)
        torch.use_deterministic_algorithms(True)

        device = torch.device("cpu")

    # ----- fast mode -----
    else:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            device = torch.device("mps")
        else:
            device = torch.device("cpu")

        # allow faster execution
        torch.use_deterministic_algorithms(False)

        if device.type == "cuda":
            torch.backends.cudnn.benchmark = True
            torch.backends.cudnn.deterministic = False

    print(f"Mode: {mode}")
    print(f"Using device: {device}")
    print("OMP_NUM_THREADS:", os.environ.get("OMP_NUM_THREADS"))
    print("MKL_NUM_THREADS:", os.environ.get("MKL_NUM_THREADS"))
    print("torch num threads:", torch.get_num_threads())
    print("torch interop threads:", torch.get_num_interop_threads())

    return device



def load_mHypothalamus(root_dir='./mHypothalamus', section_id='0.26'):
    # section id = '0.26', '0.21', '0.16', '0.11', '0.06', '0.01', '-0.04', '-0.09', '-0.14', '-0.19', '-0.24', '-0.29' 12 in total
    # cluster =     15      15      14      15      15      15      14       15       15       15      16        15
    info_file = os.path.join(root_dir, 'MERFISH_Animal1_info.xlsx')
    cnts_file = os.path.join(root_dir, 'MERFISH_Animal1_cnts.xlsx')
    xls_cnts = pd.ExcelFile(cnts_file)
    # print(xls_cnts.sheet_names)
    df_cnts = pd.read_excel(xls_cnts, section_id)
    
    xls_info = pd.ExcelFile(info_file)
    df_info = pd.read_excel(xls_info, section_id)
    # print(df_cnts, df_info)
    spatial_X = df_info.to_numpy()
    obs_ = df_info
    if len(df_info.columns) == 5:
        obs_.columns = ['psuedo_barcodes', 'x', 'y', 'original_clusters', 'Neuron_cluster_ID']
    elif len(df_info.columns) == 6:
        obs_.columns = ['psuedo_barcodes', 'x', 'y', 'cell_types', 'Neuron_cluster_ID', 'original_clusters']
        # print(section_id)
        # print(obs_['z'].nunique())
    obs_.index = obs_['psuedo_barcodes'].tolist()
    # print(obs_)

    var_ = df_cnts.iloc[:, 0]
    var_ = pd.DataFrame(var_)
    # print(var_)
    
    ad = anndata.AnnData(X=df_cnts.iloc[:,1:].T, obs=obs_, var=var_)
    ad.var.columns = ['gene_ids']
    spatial = np.vstack((ad.obs['x'].to_numpy(), ad.obs['y'].to_numpy()))
    ad.obsm['spatial'] = spatial.T
    return ad


def preprocess_for_gene_cost_merfish(adata):
    adata = adata.copy()

    sc.pp.normalize_total(adata, target_sum=1e4)

    sc.pp.log1p(adata)

    sc.pp.scale(adata, max_value=10)

    return adata

def align_genes_for_gene_cost(a1, a2):
    common = a1.var_names.intersection(a2.var_names)
    return a1[:, common].copy(), a2[:, common].copy()

# =========================================================
# Main 
# =========================================================

OUT_DIR = "./results/gene_cost_matrix"

mode = "fast"   # choose from: "reproducible", "fast"
seed = 0

device = setup_environment(mode=mode, seed=seed)
os.makedirs(OUT_DIR, exist_ok=True)

section_ids_list = [['-0.04', '-0.09']]
run_times = []

for section_ids in section_ids_list:
    dataset = section_ids[0] + '_' + section_ids[1]
    prefix = dataset.replace('.', 'p').replace('-', 'm')  
    output = '.'

    sliceA = load_mHypothalamus(section_id=section_ids[0])
    sliceB = load_mHypothalamus(section_id=section_ids[1])

    sliceA, sliceB = align_genes_for_gene_cost(sliceA, sliceB)
    sliceA = preprocess_for_gene_cost_merfish(sliceA)
    sliceB = preprocess_for_gene_cost_merfish(sliceB)

    _gene_cost_somde.compute_gene_cost(
        sliceA=sliceA,
        sliceB=sliceB,
        section_id_A=section_ids[0],
        section_id_B=section_ids[1],
        n_h=32,
        n_epoch=3500,
        lr=2e-4,
        print_step=500,
        seed=0,
        device=device,
        output_dir=OUT_DIR,
    )


Mode: fast
Using device: mps
OMP_NUM_THREADS: None
MKL_NUM_THREADS: None
torch num threads: 12
torch interop threads: 12
[0000] DGI loss = 2.6947
[0500] DGI loss = 0.0374
[1000] DGI loss = 0.0059
[1500] DGI loss = 0.0018
[2000] DGI loss = 0.0015
[2500] DGI loss = 0.0006
[3000] DGI loss = 0.0003
[3499] DGI loss = 0.0003
